# 02 — Evaluation Splits (Leakage-Free v3)

**Platform:** Kaggle Free Tier · CPU · ~3 min  
**Input:** `01-corpus-dataset` → `powershell_corpus_v3.parquet`  
**Output:** `/kaggle/working/splits/`

## Leakage fix applied here

`obfusc_*` scripts are **quarantined from all training splits** and saved as  
`test_adversarial.parquet` for Section 5.3 robustness eval only.  
Without this fix: digit_ratio and has_char_cast inflate AUROC to ≈ 0.998.

## Four-test evaluation protocol

| Split | Purpose | Malicious prevalence |
|---|---|---|
| `test_seed{N}` | Comparability with prior work | ~50% |
| `test_c2_seed{N}` | Deployment realism (C2-compliant) | ≤10% |
| `test_cross_source` | Generalisation to unseen families | ~51% |
| `test_adversarial` | Obfuscation robustness (§5.3) | 100% |

**Splits:** 70 / 15 / 15 across seeds [42, 1337, 2024]


## Cell 0 — Paths and configuration

In [1]:
import os, json, hashlib, logging, warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('splits')

SEEDS             = [42, 1337, 2024]
TEST_FRAC         = 0.15
VAL_FRAC          = 0.15
C2_MAL_PREVALENCE = 0.10
MIN_STRATUM_SIZE  = 30

OBFUSC_SOURCES = {'obfusc_encode','obfusc_compress','obfusc_string',
                  'obfusc_concat','obfusc_alias','obfusc_format','obfusc_token_HOLDOUT'}

def find_parquet(filename):
    root = Path('/kaggle/input')
    if root.exists():
        for m in root.rglob(filename): return m
    return None

CORPUS_PARQUET      = find_parquet('powershell_corpus_v3.parquet')
CROSS_SRC_PARQUET   = find_parquet('test_cross_source.parquet')
ADV_PARQUET         = find_parquet('test_adversarial.parquet')

if not CORPUS_PARQUET:
    raise FileNotFoundError(
        'powershell_corpus_v3.parquet not found. Attach 01-corpus-dataset first.')

OUT_DIR = Path('/kaggle/working/splits')
OUT_DIR.mkdir(parents=True, exist_ok=True)

log.info(f'Corpus      : {CORPUS_PARQUET}')
log.info(f'Cross-src   : {CROSS_SRC_PARQUET}')
log.info(f'Adversarial : {ADV_PARQUET}')
log.info(f'Output      : {OUT_DIR}')
print('Cell 0 OK.')


03:45:02 | INFO | Corpus      : /kaggle/input/datasets/onkarkmane1501/01-corpus-dataset/corpus/powershell_corpus_v3.parquet
03:45:02 | INFO | Cross-src   : /kaggle/input/datasets/onkarkmane1501/01-corpus-dataset/corpus/test_cross_source.parquet
03:45:02 | INFO | Adversarial : /kaggle/input/datasets/onkarkmane1501/01-corpus-dataset/corpus/test_adversarial.parquet
03:45:02 | INFO | Output      : /kaggle/working/splits


Cell 0 OK.


## Cell 1 — Load corpus and quarantine obfusc_* sources

In [2]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''): h.update(chunk)
    return h.hexdigest()

df_all = pd.read_parquet(CORPUS_PARQUET)
log.info(f'Full corpus: {len(df_all):,} records, {df_all.source.nunique()} sources')

# Schema check
for col in ['sha256','source','label','script_text','token_count']:
    if col not in df_all.columns: raise ValueError(f'Missing column: {col}')

# Dedup
before = len(df_all)
df_all = df_all.drop_duplicates(subset='sha256',keep='first').reset_index(drop=True)
if len(df_all) < before: log.warning(f'Removed {before-len(df_all):,} duplicate SHA-256s')

# Quarantine obfusc_*
is_obfusc      = df_all['source'].isin(OBFUSC_SOURCES)
df_adversarial = df_all[is_obfusc].reset_index(drop=True)
df             = df_all[~is_obfusc].reset_index(drop=True)

n_b = (df.label==0).sum(); n_m = (df.label==1).sum(); ratio = n_b/max(n_m,1)
log.info(f'Adversarial holdout : {len(df_adversarial):,} (obfusc_* sources)')
log.info(f'Clean pool          : {len(df):,}  benign={int(n_b):,}  mal={int(n_m):,}  ratio={ratio:.3f}')

# If obfusc removal created surplus benign, restore 50:50
if ratio > 1.05:
    log.info('Re-balancing to 50:50...')
    df = pd.concat([df[df.label==0].sample(n=int(n_m),random_state=42),
                    df[df.label==1]]).sample(frac=1,random_state=42).reset_index(drop=True)
    log.info(f'After rebalance: {len(df):,}')

# Load cross-source holdout and remove any SHA overlap
df_cross = None
if CROSS_SRC_PARQUET and Path(CROSS_SRC_PARQUET).exists():
    df_cross = pd.read_parquet(CROSS_SRC_PARQUET)
    overlap  = set(df['sha256']) & set(df_cross['sha256'])
    if overlap:
        log.warning(f'{len(overlap):,} SHA overlaps with cross-source — removing from pool')
        df = df[~df['sha256'].isin(overlap)].reset_index(drop=True)
    log.info(f'Cross-source: {len(df_cross):,} '
             f'(ben={(df_cross.label==0).sum()}, mal={(df_cross.label==1).sum()})')
elif ADV_PARQUET and Path(ADV_PARQUET).exists():
    # If adversarial was saved separately by NB01
    df_adversarial_ext = pd.read_parquet(ADV_PARQUET)
    df_adversarial = pd.concat([df_adversarial, df_adversarial_ext]).drop_duplicates('sha256').reset_index(drop=True)
    log.info(f'Adversarial augmented from parquet: {len(df_adversarial):,}')
else:
    log.warning('test_cross_source.parquet not found — cross-source eval will be skipped')
print('Cell 1 OK.')


03:45:09 | INFO | Full corpus: 17,396 records, 36 sources
03:45:09 | INFO | Adversarial holdout : 0 (obfusc_* sources)
03:45:09 | INFO | Clean pool          : 17,396  benign=8,698  mal=8,698  ratio=1.000
03:45:09 | INFO | Cross-source: 406 (ben=160, mal=246)


Cell 1 OK.


## Cell 2 — Stratified train / val / test splits (3 seeds)

In [3]:
source_counts = df['source'].value_counts()
large_sources = set(source_counts[source_counts >= MIN_STRATUM_SIZE].index)

def strat_key(row):
    src = row['source'] if row['source'] in large_sources else (
          'other_benign' if row['label']==0 else 'other_malicious')
    return f"{row['label']}_{src}"

df['_strat'] = df.apply(strat_key, axis=1)

# Merge strata with fewer than 6 samples
tiny_strata = df['_strat'].value_counts()
tiny_sources = set()
for key in tiny_strata[tiny_strata < 6].index:
    parts = key.split('_',1)
    if len(parts)==2: tiny_sources.add(parts[1])
if tiny_sources:
    def strat_key2(row):
        src = row['source']
        if src in tiny_sources or src not in large_sources:
            src = 'other_benign' if row['label']==0 else 'other_malicious'
        return f"{row['label']}_{src}"
    df['_strat'] = df.apply(strat_key2, axis=1)

log.info(f'Unique strata: {df._strat.nunique()}  smallest: {df._strat.value_counts().min()}')

splits_by_seed = {}
for seed in SEEDS:
    log.info(f'--- Seed {seed} ---')
    df_trainval, df_test = train_test_split(
        df, test_size=TEST_FRAC, stratify=df['_strat'], random_state=seed)
    val_frac_of_trainval = VAL_FRAC / (1 - TEST_FRAC)
    df_train, df_val = train_test_split(
        df_trainval, test_size=val_frac_of_trainval,
        stratify=df_trainval['_strat'], random_state=seed)
    for part in [df_train, df_val, df_test]:
        part.drop(columns=['_strat'],inplace=True,errors='ignore')

    # C2-compliant test: ≤10% malicious
    test_benign = df_test[df_test.label==0]
    test_mal    = df_test[df_test.label==1]
    max_mal_c2  = int(C2_MAL_PREVALENCE / (1-C2_MAL_PREVALENCE) * len(test_benign))
    n_mal_c2    = min(len(test_mal), max_mal_c2)
    df_test_c2  = pd.concat([
        test_benign,
        test_mal.sample(n=n_mal_c2, random_state=seed)
    ]).sample(frac=1, random_state=seed).reset_index(drop=True)
    actual_prev = float((df_test_c2.label==1).mean())

    log.info(f'  train={len(df_train):,}  val={len(df_val):,}  '
             f'test={len(df_test):,}  test_c2={len(df_test_c2):,} (prev={actual_prev*100:.2f}%)')
    splits_by_seed[seed] = {
        'train':   df_train.reset_index(drop=True),
        'val':     df_val.reset_index(drop=True),
        'test':    df_test.reset_index(drop=True),
        'test_c2': df_test_c2,
        'c2_meta': {'actual_prevalence': actual_prev, 'n_mal_c2': int(n_mal_c2),
                    'n_benign': int(len(test_benign)),
                    'c2_violated': bool(actual_prev > C2_MAL_PREVALENCE + 0.001)},
    }
print('Cell 2 OK.')


03:45:09 | INFO | Unique strata: 28  smallest: 34
03:45:09 | INFO | --- Seed 42 ---
03:45:09 | INFO |   train=12,176  val=2,610  test=2,610  test_c2=1,450 (prev=10.00%)
03:45:09 | INFO | --- Seed 1337 ---
03:45:09 | INFO |   train=12,176  val=2,610  test=2,610  test_c2=1,450 (prev=10.00%)
03:45:09 | INFO | --- Seed 2024 ---
03:45:09 | INFO |   train=12,176  val=2,610  test=2,610  test_c2=1,450 (prev=10.00%)


Cell 2 OK.


## Cell 3 — Class weights

In [4]:
class_weights_per_seed = {}
for seed in SEEDS:
    train = splits_by_seed[seed]['train']
    n_neg = int((train.label==0).sum())
    n_pos = int((train.label==1).sum())
    class_weights_per_seed[str(seed)] = {
        'n_train':          n_pos + n_neg,
        'n_benign':         n_neg,
        'n_malicious':      n_pos,
        'pos_weight_bce':   round(n_neg/max(n_pos,1), 6),
        'scale_pos_weight': round(n_neg/max(n_pos,1), 6),
        'focal_alpha':      round(n_pos/(n_pos+n_neg), 6),
        'class_weight_0':   round(n_pos/(n_pos+n_neg), 6),
        'class_weight_1':   round(n_neg/(n_pos+n_neg), 6),
    }
    log.info(f'Seed {seed}: n_benign={n_neg:,}  n_malicious={n_pos:,}  '
             f'scale_pos_weight={class_weights_per_seed[str(seed)]["scale_pos_weight"]:.4f}')
print('Cell 3 OK.')


03:45:09 | INFO | Seed 42: n_benign=6,089  n_malicious=6,087  scale_pos_weight=1.0003
03:45:09 | INFO | Seed 1337: n_benign=6,089  n_malicious=6,087  scale_pos_weight=1.0003
03:45:09 | INFO | Seed 2024: n_benign=6,089  n_malicious=6,087  scale_pos_weight=1.0003


Cell 3 OK.


## Cell 4 — Save all files

In [5]:
saved_files = {}

for seed in SEEDS:
    for split_name in ['train','val','test','test_c2']:
        fname = OUT_DIR / f'{split_name}_seed{seed}.parquet'
        splits_by_seed[seed][split_name].to_parquet(fname, index=False, engine='pyarrow')
        saved_files[f'{split_name}_seed{seed}'] = fname
        log.info(f'Saved {fname.name}  ({fname.stat().st_size/1e3:.0f} KB)')

if df_cross is not None:
    cross_path = OUT_DIR / 'test_cross_source.parquet'
    df_cross.to_parquet(cross_path, index=False, engine='pyarrow')
    saved_files['test_cross_source'] = cross_path

adv_out = OUT_DIR / 'test_adversarial.parquet'
df_adversarial.to_parquet(adv_out, index=False, engine='pyarrow')
saved_files['test_adversarial'] = adv_out

weights_path = OUT_DIR / 'class_weights_per_seed.json'
with open(weights_path,'w') as f: json.dump(class_weights_per_seed, f, indent=2)

manifest = {
    'created_utc':             datetime.now(timezone.utc).isoformat(),
    'splits_version':          'v3',
    'leakage_fix':             'obfusc_* quarantined to test_adversarial.parquet',
    'obfusc_sources_removed':  sorted(OBFUSC_SOURCES),
    'seeds': SEEDS, 'test_frac': TEST_FRAC,
    'val_frac': VAL_FRAC, 'c2_prevalence_cap': C2_MAL_PREVALENCE,
    'splits': {
        str(seed): {
            sn: {'n': int(len(splits_by_seed[seed][sn])),
                 'n_benign':    int((splits_by_seed[seed][sn].label==0).sum()),
                 'n_malicious': int((splits_by_seed[seed][sn].label==1).sum())}
            for sn in ['train','val','test','test_c2']
        } for seed in SEEDS
    },
}
with open(OUT_DIR/'split_manifest.json','w') as f: json.dump(manifest,f,indent=2)
log.info(f'Manifest saved: {OUT_DIR}/split_manifest.json')
print('Cell 4 OK. All files saved.')


03:45:13 | INFO | Saved train_seed42.parquet  (348881 KB)
03:45:14 | INFO | Saved val_seed42.parquet  (93374 KB)
03:45:14 | INFO | Saved test_seed42.parquet  (68350 KB)
03:45:14 | INFO | Saved test_c2_seed42.parquet  (7072 KB)
03:45:18 | INFO | Saved train_seed1337.parquet  (349506 KB)
03:45:18 | INFO | Saved val_seed1337.parquet  (78846 KB)
03:45:19 | INFO | Saved test_seed1337.parquet  (82142 KB)
03:45:19 | INFO | Saved test_c2_seed1337.parquet  (6594 KB)
03:45:23 | INFO | Saved train_seed2024.parquet  (368849 KB)
03:45:23 | INFO | Saved val_seed2024.parquet  (79748 KB)
03:45:24 | INFO | Saved test_seed2024.parquet  (61918 KB)
03:45:24 | INFO | Saved test_c2_seed2024.parquet  (11352 KB)
03:45:24 | INFO | Manifest saved: /kaggle/working/splits/split_manifest.json


Cell 4 OK. All files saved.


## Cell 5 — Sanity checks (all must pass before running NB03)

In [6]:
checks = []
def chk(cond, name, detail=''):
    checks.append((bool(cond), name, str(detail)))

# 1. No obfusc_* in any training split
for seed in SEEDS:
    train_sources = set(splits_by_seed[seed]['train']['source'].unique())
    leaked = train_sources & OBFUSC_SOURCES
    chk(len(leaked)==0, f'Seed {seed}: no obfusc_* in train', str(leaked or 'none'))

# 2. No SHA leakage across train/val/test
for seed in SEEDS:
    tr = set(splits_by_seed[seed]['train']['sha256'])
    va = set(splits_by_seed[seed]['val']['sha256'])
    te = set(splits_by_seed[seed]['test']['sha256'])
    overlap = (tr & va)|(tr & te)|(va & te)
    chk(len(overlap)==0, f'Seed {seed}: no SHA leakage', f'{len(overlap)} overlaps')

# 3. C2 test ⊆ balanced test
for seed in SEEDS:
    tc2 = set(splits_by_seed[seed]['test_c2']['sha256'])
    te  = set(splits_by_seed[seed]['test']['sha256'])
    chk(tc2.issubset(te), f'Seed {seed}: test_c2 ⊆ test')

# 4. C2 prevalence ≤10%
for seed in SEEDS:
    meta = splits_by_seed[seed]['c2_meta']
    chk(not meta['c2_violated'], f'Seed {seed}: C2 prevalence ≤10%',
        f'actual={meta["actual_prevalence"]*100:.2f}%')

# 5. Train ≈50:50
for seed in SEEDS:
    mal_frac = float((splits_by_seed[seed]['train'].label==1).mean())
    chk(0.45<=mal_frac<=0.55, f'Seed {seed}: train ≈50:50', f'{mal_frac*100:.1f}% mal')

# 6. Output files exist
for seed in SEEDS:
    for sn in ['train','val','test','test_c2']:
        chk((OUT_DIR/f'{sn}_seed{seed}.parquet').exists(), f'{sn}_seed{seed}.parquet exists')
chk((OUT_DIR/'class_weights_per_seed.json').exists(), 'class_weights_per_seed.json exists')
chk((OUT_DIR/'split_manifest.json').exists(), 'split_manifest.json exists')

all_ok = True
for ok, name, detail in checks:
    print(f'  {"[OK]" if ok else "[FAIL]":<8} {name}')
    if detail: print(f'           {detail}')
    if not ok: all_ok = False

n_pass = sum(1 for ok,_,_ in checks if ok)
n_fail = len(checks)-n_pass
print(f'\nResults: {n_pass} passed, {n_fail} failed')
if all_ok:
    print('\nALL CHECKS PASSED.')
    print('Next: upload /kaggle/working/splits/ as Kaggle Dataset "02-splits-dataset"')
    print('Then run 03_triage.ipynb')
else:
    raise RuntimeError(f'{n_fail} sanity checks FAILED.')


  [OK]     Seed 42: no obfusc_* in train
           none
  [OK]     Seed 1337: no obfusc_* in train
           none
  [OK]     Seed 2024: no obfusc_* in train
           none
  [OK]     Seed 42: no SHA leakage
           0 overlaps
  [OK]     Seed 1337: no SHA leakage
           0 overlaps
  [OK]     Seed 2024: no SHA leakage
           0 overlaps
  [OK]     Seed 42: test_c2 ⊆ test
  [OK]     Seed 1337: test_c2 ⊆ test
  [OK]     Seed 2024: test_c2 ⊆ test
  [OK]     Seed 42: C2 prevalence ≤10%
           actual=10.00%
  [OK]     Seed 1337: C2 prevalence ≤10%
           actual=10.00%
  [OK]     Seed 2024: C2 prevalence ≤10%
           actual=10.00%
  [OK]     Seed 42: train ≈50:50
           50.0% mal
  [OK]     Seed 1337: train ≈50:50
           50.0% mal
  [OK]     Seed 2024: train ≈50:50
           50.0% mal
  [OK]     train_seed42.parquet exists
  [OK]     val_seed42.parquet exists
  [OK]     test_seed42.parquet exists
  [OK]     test_c2_seed42.parquet exists
  [OK]     train_seed133